# Approach D — Interactive Case Viewer\n\nCompares MRI · Ground Truth · Prediction for test cases.  \nUse the sliders to scroll slices; use the dropdowns to switch cases and view axis."

In [ ]:
from pathlib import Path
import numpy as np
import nibabel as nib

REPO     = Path("..").resolve()
PRED_DIR = REPO / "approach_d/predictions"
TEST_TXT = REPO / "data/test.txt"

gt_map = {}
for line in TEST_TXT.read_text().splitlines()[1:]:
    line = line.strip()
    if not line: continue
    p = line.split(",")
    stem = Path(p[0]).name.replace(".nii.gz", "")
    gt_map[stem] = (Path(p[0]), Path(p[1]))

def _load(path):
    return np.asarray(nib.load(str(path)).dataobj).astype(np.float32)

def _norm(v):
    lo, hi = np.percentile(v, 1), np.percentile(v, 99)
    return np.clip((v - lo) / max(hi - lo, 1e-6), 0, 1)

def overlay(gray, mask, color, alpha=0.45):
    rgb = np.stack([gray]*3, axis=-1).copy()
    if mask.any():
        rgb[mask > 0] = (1-alpha)*rgb[mask > 0] + alpha*np.array(color, dtype=np.float32)
    return rgb

def dice(pred, gt):
    p, g = pred > 0, gt > 0
    d = p.sum() + g.sum()
    return float(2*(p&g).sum()/d) if d > 0 else 1.0

# pre-compute best slice index per axis at cache time (fast)
_cache = {}
def get_data(name):
    if name not in _cache:
        img_p, gt_p = gt_map[name]
        pred_p = PRED_DIR / f"{name}.nii.gz"
        img  = _norm(_load(img_p))
        gt   = (_load(gt_p) > 0).astype(np.uint8)
        pred = (_load(pred_p) > 0).astype(np.uint8) if pred_p.exists() else np.zeros_like(gt)
        # best slice per axis: argmax of GT sum (computed once)
        best = {}
        for ax in range(3):
            sums = gt.sum(axis=tuple(i for i in range(3) if i != ax))
            best[ax] = int(np.argmax(sums)) if sums.max() > 0 else img.shape[ax] // 2
        dc    = dice(pred, gt)
        n_gt  = int(gt.sum())
        n_pred = int(pred.sum())
        _cache[name] = dict(img=img, gt=gt, pred=pred, best=best,
                            dc=dc, n_gt=n_gt, n_pred=n_pred)
    return _cache[name]

def get_slice(vol, axis, idx):
    s = int(np.clip(idx, 0, vol.shape[axis]-1))
    if axis == 0: return np.rot90(vol[s])
    if axis == 1: return np.rot90(vol[:, s])
    return np.rot90(vol[:, :, s])

AXIS_MAP = {"Axial (Z)": 0, "Coronal (Y)": 1, "Sagittal (X)": 2}
print(f"Loaded {len(gt_map)} test cases. Helpers ready.")

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

HIGHLIGHT = ["NU28", "NU188", "EMC077", "NU71", "NYU0165"]
DEFAULT_CASES = [c for c in HIGHLIGHT if c in gt_map]

# ── widgets ───────────────────────────────────────────────────────────────────
case_dd  = widgets.Dropdown(options=DEFAULT_CASES, value=DEFAULT_CASES[0],
                            description="Case:", layout=widgets.Layout(width="180px"))
axis_dd  = widgets.Dropdown(options=list(AXIS_MAP.keys()), value="Axial (Z)",
                            description="View:", layout=widgets.Layout(width="190px"))
slice_sl = widgets.IntSlider(min=0, max=1, value=0, description="Slice",
                             continuous_update=True,   # fast now — just set_data()
                             layout=widgets.Layout(width="500px"))

# ── figure created ONCE — only pixel data is updated on slider drag ───────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.patch.set_facecolor("#111")
plt.subplots_adjust(left=0.01, right=0.99, top=0.88, bottom=0.02, wspace=0.04)

blank = np.zeros((10, 10, 3), dtype=np.float32)
im_objs = [ax.imshow(blank, vmin=0, vmax=1, interpolation="bilinear") for ax in axes]
for ax, t in zip(axes, ["MRI", "GT  (green)", "Prediction  (orange)"]):
    ax.set_title(t, color="white", fontsize=10)
    ax.axis("off"); ax.set_facecolor("black")
title_obj = fig.suptitle("", color="white", fontsize=11)

def _redraw(name, axis_label, sl):
    axis = AXIS_MAP[axis_label]
    d = get_data(name)
    sl = int(np.clip(sl, 0, d["img"].shape[axis]-1))

    gray = get_slice(d["img"],  axis, sl)
    gtm  = get_slice(d["gt"],   axis, sl)
    prm  = get_slice(d["pred"], axis, sl)

    im_objs[0].set_data(np.stack([gray]*3, axis=-1))
    im_objs[1].set_data(overlay(gray, gtm,  [0.0, 1.0, 0.0]))
    im_objs[2].set_data(overlay(gray, prm,  [1.0, 0.35, 0.0]))

    # rescale each axes to the new image shape
    h, w = gray.shape
    for im in im_objs:
        im.set_extent([-0.5, w-0.5, h-0.5, -0.5])

    missed = "  ⚠ model predicted nothing!" if d["n_pred"]==0 and d["n_gt"]>0 else ""
    title_obj.set_text(
        f"{name}  |  {axis_label}  sl {sl}  |  "
        f"Dice {d['dc']:.4f}  |  GT {d['n_gt']} vox  |  Pred {d['n_pred']} vox{missed}"
    )
    fig.canvas.draw_idle()   # ← only flushes changed pixels, very fast

def on_case_axis(_):
    name = case_dd.value; axis = AXIS_MAP[axis_dd.value]
    d = get_data(name)
    slice_sl.max   = d["img"].shape[axis] - 1
    # unobserve temporarily to avoid double-fire
    slice_sl.unobserve(on_slice, names="value")
    slice_sl.value = d["best"][axis]
    slice_sl.observe(on_slice, names="value")
    _redraw(name, axis_dd.value, slice_sl.value)

def on_slice(change):
    _redraw(case_dd.value, axis_dd.value, change["new"])

case_dd.observe(on_case_axis, names="value")
axis_dd.observe(on_case_axis, names="value")
slice_sl.observe(on_slice,    names="value")

on_case_axis(None)   # initial draw
display(widgets.HBox([case_dd, axis_dd]), slice_sl)

## Browse any test case\nChange the `CASES` list below to any subset of the 74 test cases."

In [ ]:
CASES = list(gt_map.keys())   # all 74 — or replace with any subset

case_dd2  = widgets.Dropdown(options=CASES, value=CASES[0],
                             description="Case:", layout=widgets.Layout(width="180px"))
axis_dd2  = widgets.Dropdown(options=list(AXIS_MAP.keys()), value="Axial (Z)",
                             description="View:", layout=widgets.Layout(width="190px"))
slice_sl2 = widgets.IntSlider(min=0, max=1, value=0, description="Slice",
                              continuous_update=True,
                              layout=widgets.Layout(width="500px"))

fig2, axes2 = plt.subplots(1, 3, figsize=(14, 4.5))
fig2.patch.set_facecolor("#111")
plt.subplots_adjust(left=0.01, right=0.99, top=0.88, bottom=0.02, wspace=0.04)

im_objs2 = [ax.imshow(np.zeros((10,10,3), dtype=np.float32), vmin=0, vmax=1,
                      interpolation="bilinear") for ax in axes2]
for ax, t in zip(axes2, ["MRI", "GT  (green)", "Prediction  (orange)"]):
    ax.set_title(t, color="white", fontsize=10)
    ax.axis("off"); ax.set_facecolor("black")
title2 = fig2.suptitle("", color="white", fontsize=11)

def _redraw2(name, axis_label, sl):
    axis = AXIS_MAP[axis_label]
    d = get_data(name)
    sl = int(np.clip(sl, 0, d["img"].shape[axis]-1))
    gray = get_slice(d["img"], axis, sl)
    gtm  = get_slice(d["gt"],  axis, sl)
    prm  = get_slice(d["pred"],axis, sl)
    im_objs2[0].set_data(np.stack([gray]*3, axis=-1))
    im_objs2[1].set_data(overlay(gray, gtm,  [0.0, 1.0, 0.0]))
    im_objs2[2].set_data(overlay(gray, prm,  [1.0, 0.35, 0.0]))
    h, w = gray.shape
    for im in im_objs2: im.set_extent([-0.5, w-0.5, h-0.5, -0.5])
    missed = "  ⚠ model predicted nothing!" if d["n_pred"]==0 and d["n_gt"]>0 else ""
    title2.set_text(f"{name}  |  {axis_label}  sl {sl}  |  Dice {d['dc']:.4f}  |  "
                    f"GT {d['n_gt']} vox  |  Pred {d['n_pred']} vox{missed}")
    fig2.canvas.draw_idle()

def on_ca2(_):
    name = case_dd2.value; axis = AXIS_MAP[axis_dd2.value]
    d = get_data(name)
    slice_sl2.max = d["img"].shape[axis] - 1
    slice_sl2.unobserve(on_sl2, names="value")
    slice_sl2.value = d["best"][axis]
    slice_sl2.observe(on_sl2, names="value")
    _redraw2(name, axis_dd2.value, slice_sl2.value)

def on_sl2(change):
    _redraw2(case_dd2.value, axis_dd2.value, change["new"])

case_dd2.observe(on_ca2, names="value")
axis_dd2.observe(on_ca2, names="value")
slice_sl2.observe(on_sl2, names="value")

on_ca2(None)
display(widgets.HBox([case_dd2, axis_dd2]), slice_sl2)